# Manipulation Check: Patient-Centred vs Task-Centred Linguistic Style

Verifies that the LLM prompt manipulation produced measurable linguistic differences in the
AI doctor's speech, and that clinical content was held constant across conditions.

**Outputs**
1. `manipulation_check_features.csv` — per-participant linguistic features (merge into `PREPROCESSED.csv` as covariates)
2. `manipulation_check_table.tex` — LaTeX table for the appendix
3. `content_equivalence.csv` — per-participant flags for whether the same diagnosis and treatment were delivered
4. `distinctive_terms.csv` — words most characteristic of each condition

**Design logic**

- Only **assistant** turns are analysed. The manipulation is the AI's language.
- **Fixed utterances** (the scripted greeting, `Scanning in progress...`) are excluded. They are identical by design and dilute the contrast.
- **Dosage confounds** (word count, turn count, duration) are reported *separately and first*. If PC talked substantially more than TC, H2 has its own confound and any style effect is uninterpretable without covarying it.
- **Content equivalence** is checked before style. If the two conditions delivered different clinical content, the claim that style operates independently of the advice does not hold.

In [1]:
import json, re, glob, os, math, warnings
from collections import Counter
import numpy as np
import pandas as pd
from scipy import stats

In [2]:
warnings.filterwarnings("ignore")
pd.set_option("display.width", 200, "display.max_columns", 60)

In [3]:
# ---- CONFIGURE ----------------------------------------------------------
TRANSCRIPT_DIR = '/Users/jhy33/Desktop/AIDocTrustAnalysis/LLM Conversation History'   # folder containing LLMconvo_ppt*.json
OUT_DIR        = '/Users/jhy33/Desktop/AIDocTrustAnalysis/transcript output'
os.makedirs(OUT_DIR, exist_ok=True)

# Map the condition CODE inside each JSON to your paper's labels.
# NOTE: these codes come from the JSON files and are NOT necessarily the same
# integers as the `condition` column in PREPROCESSED.csv. Check before merging.
CONDITION_MAP = {
    "1": "Patient-Centred",   # LLM_Empathetic
    "2": "Task-Centred",      # LLM_Professional
}

# Utterances identical across conditions by design -> excluded from style analysis.
FIXED_UTTERANCES = {
    "hi, i am your ai doctor. how are you today?",
    "scanning in progress...",
}
# -------------------------------------------------------------------------
print("ready")

ready


# load transcripts

turns sorted by timestamp

`n_order_anomalies` records how often logged order is non-monotonic

In [8]:
def load_transcripts(path):
    rows, meta = [], []
    files = sorted(glob.glob(os.path.join(path, "*.json")))
    if not files:
        raise FileNotFoundError(f"No .json files in {path}")

    skipped_files = []
    for fp in files:
        try:
            with open(fp, "r", encoding="utf-8") as f:
                d = json.load(f)
        except json.JSONDecodeError as exc:
            skipped_files.append((fp, exc))
            continue

        pid  = str(d.get("participant_id", os.path.basename(fp)))
        cond = d.get("condition", {})
        code_ = str(cond.get("code", "NA"))

        turns = d.get("transcript", [])
        ts = []
        for t in turns:
            try:
                ts.append(pd.to_datetime(t.get("timestamp"), utc=True, format="mixed"))
            except Exception:
                ts.append(pd.NaT)

        anomalies = sum(1 for a, b in zip(ts, ts[1:])
                        if pd.notna(a) and pd.notna(b) and b < a)

        order = np.argsort([t.value if pd.notna(t) else i
                            for i, t in enumerate(ts)], kind="stable")

        for new_i, old_i in enumerate(order):
            t = turns[old_i]
            rows.append({
                "PID": pid, "cond_code": code_,
                "condition": CONDITION_MAP.get(code_, cond.get("name", "Unknown")),
                "turn_index": new_i,
                "role": t.get("role", ""),
                "text": (t.get("content") or "").strip(),
                "timestamp": ts[old_i],
            })

        try:
            dur = (pd.to_datetime(d["session_end"]) - pd.to_datetime(d["session_start"])).total_seconds() / 60
        except Exception:
            dur = np.nan

        meta.append({"PID": pid, "cond_code": code_,
                     "condition": CONDITION_MAP.get(code_, cond.get("name", "Unknown")),
                     "session_minutes": dur,
                     "model": d.get("model", "NA"),
                     "n_order_anomalies": anomalies,
                     "file": os.path.basename(fp)})

    for fp, exc in skipped_files:
        print(f"Skipped invalid JSON: {os.path.basename(fp)} (line {exc.lineno}, column {exc.colno}: {exc.msg})")
    if not rows:
        raise ValueError(f"No valid transcripts found in {path}")

    return pd.DataFrame(rows), pd.DataFrame(meta)


turns_df, meta_df = load_transcripts(TRANSCRIPT_DIR)

# Flag fixed/scripted utterances
turns_df["is_fixed"] = turns_df["text"].str.lower().str.strip().isin(FIXED_UTTERANCES)

print(f"{meta_df.PID.nunique()} participants | {len(turns_df)} turns")
print(meta_df.groupby("condition").agg(n=("PID", "nunique"),
                                       mean_minutes=("session_minutes", "mean"),
                                       anomalies=("n_order_anomalies", "sum")))

106 participants | 3169 turns
                  n  mean_minutes  anomalies
condition                                   
Patient-Centred  53      9.751753         51
Task-Centred     53      9.432241         43


# lexicons & patterns



In [9]:
# --- Person deixis ---------------------------------------------------------
FIRST_SG  = r"\b(i|i'm|i've|i'll|i'd|me|my|mine|myself)\b"
FIRST_PL  = r"\b(we|we'll|we've|we're|us|our|ours|let's)\b"
SECOND    = r"\b(you|you're|you've|you'll|you'd|your|yours|yourself)\b"

# --- Agentless / passive constructions (hallmark of the clinical register) ---
PASSIVE = r"\b(has|have|had|is|are|was|were|been|being)\s+(been\s+)?" \
          r"(\w+ed|reviewed|recorded|noted|confirmed|ruled|required|completed|" \
          r"considered|scheduled|based|conducted|performed|advised|prescribed)\b"

# --- Affective / relational vocabulary -------------------------------------
AFFECT = {
    "glad","happy","pleased","sorry","understand","understandable","appreciate","thank",
    "thanks","helpful","comfortable","comforting","reassure","reassuring","difficult",
    "hard","frustrating","distressing","uncomfortable","worry","worried","concern",
    "concerned","support","supportive","care","caring","alone","together","wish","well",
    "excellent","great","wonderful","absolutely","fine","okay","hope","hopeful","manage",
    "manageable","gentle","kind","feel","feeling","feelings","take care","of course",
}

# --- Acknowledgement style: warm vs procedural ------------------------------
ACK_WARM = [
    r"thank you for (sharing|letting me know|telling me)", r"that'?s helpful",
    r"i'?m glad", r"good to know", r"thanks for", r"that'?s absolutely fine",
    r"that'?s (completely )?(fine|okay)", r"i understand", r"you'?re not alone",
    r"i hear you", r"that (sounds|must be)",
]
ACK_PROCEDURAL = [
    r"\bnoted\b", r"has been (recorded|noted|reviewed|logged)", r"\bunderstood\b",
    r"\bi see\b", r"that has been", r"\bconfirmed\b", r"\bas per\b",
    r"\bnext:", r"\brecorded\b",
]

# --- Hedges / mitigated requests -------------------------------------------
HEDGE = [
    r"\bcould you\b", r"\bwould you\b", r"\bif you'?d like\b", r"\bif you want\b",
    r"\bperhaps\b", r"\bmaybe\b", r"\bmight\b", r"\ba (bit|little)\b",
    r"\bwe can\b", r"\blet'?s\b", r"\bwhenever you'?re ready\b",
    r"\bdo you feel\b", r"\bwould you like\b", r"\bshall we\b", r"\bwhen you'?re ready\b",
]

# --- Bare directives --------------------------------------------------------
DIRECTIVE = [
    r"(?:^|(?<=[.!?]\s))please\s+\w+", r"(?:^|(?<=[.!?]\s))(present|hold|confirm|"
    r"provide|state|apply|proceed|continue|answer|report|describe|list)\b",
]

# --- Emotion-eliciting questions -------------------------------------------
EMOTION_Q = [
    r"how (are|do) you feel", r"how are you feeling", r"anything (else )?on your mind",
    r"how does that (sound|feel)", r"are you (comfortable|worried|concerned)",
    r"what (are your|concerns)", r"does that (sound|feel)",
]

# --- Explicit personalisation: referring back to the patient's own account --
PERSONALISATION = [
    r"you (mentioned|said|told me|noticed|described|shared|reported)",
    r"your (symptoms|history|hands|skin|itching|sleep|case|concerns|situation|experience)",
    r"(based on|from) (your|what you)", r"you'?ve (been|noticed|experienced|had)",
    r"for you\b", r"in your case\b",
]

# --- Technical / biomedical register ---------------------------------------
TECHNICAL = {
    "diagnosis","diagnostic","inflammation","inflammatory","dermatitis","psoriasis",
    "eczema","topical","steroid","corticosteroid","potency","high-potency","epidermal",
    "barrier","lesion","lesions","erythema","pruritus","aetiology","etiology","emollient",
    "moisturiser","moisturizer","fingertip","unit","regimen","protocol","database",
    "analysis","assessment","examination","scan","parameters","criteria","differential",
    "contraindication","adherence","prophylaxis","recurrence",
}

# --- Function words for the echo measure -----------------------------------
STOPWORDS = set("""a an the and or but if of to in on at for with without from by as is are
was were be been being do does did doing have has had having i you he she it we they me my
your his her its our their this that these those there here what which who whom how when
why not no yes so very just really quite too also then than can could would should may might
will shall must about into over under again more most some any all each other m s t re ve ll
d um uh mm mmhmm yeah ok okay""".split())

TOKEN_RE = re.compile(r"[a-z][a-z'\-]+")
print("lexicons loaded")

lexicons loaded


# feature extraction

In [10]:
def tokens(text):
    return TOKEN_RE.findall(text.lower())

def count_patterns(text, patterns):
    t = text.lower()
    return sum(len(re.findall(p, t)) for p in patterns)

def syllables(word):
    word = word.lower().strip(".:;?!")
    if not word:
        return 0
    v = "aeiouy"
    n, prev = 0, False
    for ch in word:
        isv = ch in v
        if isv and not prev:
            n += 1
        prev = isv
    if word.endswith("e") and n > 1:
        n -= 1
    return max(n, 1)

def flesch_kincaid(text):
    sents = [s for s in re.split(r"[.!?]+", text) if s.strip()]
    words = tokens(text)
    if not sents or not words:
        return np.nan
    syl = sum(syllables(w) for w in words)
    return 0.39 * (len(words) / len(sents)) + 11.8 * (syl / len(words)) - 15.59


def extract_features(pid_turns):
    """pid_turns: turns for ONE participant, timestamp-sorted."""
    a = pid_turns[(pid_turns.role == "assistant") & (~pid_turns.is_fixed)]
    u = pid_turns[pid_turns.role == "user"]

    text = " ".join(a.text.tolist())
    toks = tokens(text)
    n = len(toks)
    per100 = lambda x: 100 * x / n if n else np.nan

    # --- patient echo: assistant content words that appeared in EARLIER user turns
    echo_hits = echo_total = 0
    seen_user = set()
    for _, row in pid_turns.iterrows():
        ct = {w for w in tokens(row.text) if w not in STOPWORDS and len(w) > 2}
        if row.role == "user":
            seen_user |= ct
        elif not row.is_fixed:
            echo_total += len(ct)
            echo_hits  += len(ct & seen_user)

    f = {
        # ---- DOSAGE (confound controls, not style measures) ----
        "n_assistant_turns":  len(a),
        "assistant_words":    n,
        "mean_words_per_turn": n / len(a) if len(a) else np.nan,
        "n_user_turns":       len(u),
        "user_words":         len(tokens(" ".join(u.text.tolist()))),
        "n_questions":        text.count("?"),
        "type_token_ratio":   len(set(toks)) / n if n else np.nan,
        "fk_grade":           flesch_kincaid(text),

        # ---- PATIENT-CENTRED markers (rate per 100 assistant words) ----
        "first_sg_p100":        per100(len(re.findall(FIRST_SG,  text.lower()))),
        "first_pl_p100":        per100(len(re.findall(FIRST_PL,  text.lower()))),
        "second_person_p100":   per100(len(re.findall(SECOND,    text.lower()))),
        "affect_p100":          per100(sum(1 for w in toks if w in AFFECT)),
        "ack_warm_p100":        per100(count_patterns(text, ACK_WARM)),
        "hedge_p100":           per100(count_patterns(text, HEDGE)),
        "emotion_q_p100":       per100(count_patterns(text, EMOTION_Q)),
        "personalisation_p100": per100(count_patterns(text, PERSONALISATION)),
        "patient_echo_rate":    echo_hits / echo_total if echo_total else np.nan,

        # ---- TASK-CENTRED markers ----
        "passive_p100":         per100(len(re.findall(PASSIVE, text.lower()))),
        "ack_procedural_p100":  per100(count_patterns(text, ACK_PROCEDURAL)),
        "directive_p100":       per100(count_patterns(text, DIRECTIVE)),
        "technical_p100":       per100(sum(1 for w in toks if w in TECHNICAL)),
    }
    return f


feat_rows = []
for pid, g in turns_df.groupby("PID", sort=False):
    g = g.sort_values("turn_index")
    row = {"PID": pid, "condition": g.condition.iloc[0]}
    row.update(extract_features(g))
    feat_rows.append(row)

features = pd.DataFrame(feat_rows).merge(
    meta_df[["PID", "session_minutes", "n_order_anomalies"]], on="PID", how="left")

# Composite style index: PC markers minus TC markers, on z-scores
PC_COLS = ["first_sg_p100","first_pl_p100","second_person_p100","affect_p100",
           "ack_warm_p100","hedge_p100","emotion_q_p100","personalisation_p100"]
TC_COLS = ["passive_p100","ack_procedural_p100","directive_p100","technical_p100"]

z = lambda s: (s - s.mean()) / s.std(ddof=1) if s.std(ddof=1) > 0 else s * 0
features["PC_index"]    = features[PC_COLS].apply(z).mean(axis=1)
features["TC_index"]    = features[TC_COLS].apply(z).mean(axis=1)
features["style_index"] = features["PC_index"] - features["TC_index"]

features.head()

,PID,condition,n_assistant_turns,assistant_words,mean_words_per_turn,n_user_turns,user_words,n_questions,type_token_ratio,fk_grade,first_sg_p100,first_pl_p100,second_person_p100,affect_p100,ack_warm_p100,hedge_p100,emotion_q_p100,personalisation_p100,patient_echo_rate,passive_p100,ack_procedural_p100,directive_p100,technical_p100,session_minutes,n_order_anomalies,PC_index,TC_index,style_index
0,002,Patient-Centred,12,314,26.166667,11,43,11,0.503185,4.457976,2.547771,4.140127,6.369427,4.777070,1.273885,1.910828,1.910828,0.955414,0.035088,0.636943,0.318471,0.318471,5.414013,9.013480,1,1.143276,-0.500346,1.643623
1,003,Task-Centred,10,224,22.400000,9,22,9,0.558036,8.366912,0.446429,0.446429,2.232143,1.339286,0.000000,0.000000,0.000000,0.446429,0.000000,2.678571,6.250000,0.000000,11.607143,7.904059,0,-1.188667,0.896703,-2.085370
2,005,Patient-Centred,16,428,26.750000,19,160,14,0.478972,4.763172,2.336449,3.738318,4.906542,3.971963,1.869159,1.168224,2.336449,0.700935,0.085586,0.467290,0.467290,0.000000,3.504673,9.982399,2,0.913023,-1.090833,2.003857
3,6,Task-Centred,17,402,23.647059,15,51,15,0.383085,8.073038,2.238806,0.497512,2.736318,1.741294,0.000000,0.248756,0.000000,0.497512,0.025424,2.985075,5.472637,0.248756,12.935323,10.045919,0,-0.872429,1.314729,-2.187158
4,008,Patient-Centred,17,402,23.647059,17,55,15,0.519900,3.862811,3.482587,3.233831,6.716418,5.970149,1.741294,2.487562,2.238806,1.492537,0.047847,0.248756,0.248756,0.000000,2.985075,9.162229,0,1.589539,-1.220111,2.809650


## Content equivalence

The prerequisite for every claim in the paper about style operating *independently of the
clinical advice*. If these proportions are not near 1.00 in both conditions, weaken the claim
to "independent of the diagnosis label" or fix the stimuli.

In [11]:
CONTENT_CHECKS = {
    "dx_eczema":        r"\b(hand )?eczema\b",
    "dx_barrier":       r"(skin )?barrier",
    "dx_inflammatory":  r"inflammat",
    "ruled_out_derm":   r"contact dermatitis",
    "ruled_out_psor":   r"psoriasis",
    "tx_steroid":       r"(topical )?(cortico)?steroid",
    "tx_potency":       r"high[- ]potency",
    "tx_duration":      r"\b7 days\b|\bseven days\b",
    "tx_ftu":           r"fingertip unit",
    "tx_frequency":     r"once daily|once a day",
    "tx_moisturiser":   r"moisturi[sz]er",
    "tx_alternatives":  r"alternative treatment",
    "followup":         r"(1|one) to (2|two) weeks?",
}

rows = []
for pid, g in turns_df.groupby("PID", sort=False):
    text = " ".join(g[g.role == "assistant"].text.tolist()).lower()
    r = {"PID": pid, "condition": g.condition.iloc[0]}
    r.update({k: int(bool(re.search(p, text))) for k, p in CONTENT_CHECKS.items()})
    rows.append(r)

content = pd.DataFrame(rows)
content["n_elements_delivered"] = content[list(CONTENT_CHECKS)].sum(axis=1)

summary = content.groupby("condition")[list(CONTENT_CHECKS)].mean().T
summary["difference"] = summary.iloc[:, 0] - summary.iloc[:, 1] if summary.shape[1] > 1 else np.nan
print("Proportion of consultations in which each clinical element was delivered:\n")
print(summary.round(3))

print("\nMean number of the {} elements delivered:".format(len(CONTENT_CHECKS)))
print(content.groupby("condition").n_elements_delivered.agg(["mean", "std", "min"]).round(2))

flag = content[content.n_elements_delivered < len(CONTENT_CHECKS) - 2]
if len(flag):
    print(f"\n*** {len(flag)} consultations missing 3+ elements — inspect these ***")
    print(flag[["PID", "condition", "n_elements_delivered"]].to_string(index=False))

Proportion of consultations in which each clinical element was delivered:

condition        Patient-Centred  Task-Centred  difference
dx_eczema                  1.000         0.981       0.019
dx_barrier                 1.000         0.981       0.019
dx_inflammatory            0.943         1.000      -0.057
ruled_out_derm             0.849         0.830       0.019
ruled_out_psor             0.849         0.830       0.019
tx_steroid                 1.000         1.000       0.000
tx_potency                 1.000         0.981       0.019
tx_duration                1.000         0.981       0.019
tx_ftu                     1.000         0.981       0.019
tx_frequency               1.000         0.981       0.019
tx_moisturiser             1.000         0.981       0.019
tx_alternatives            0.830         0.868      -0.038
followup                   0.811         0.887      -0.075

Mean number of the 13 elements delivered:
                  mean   std  min
condition             

# conversation structure / volume
word count, turn count, duration

In [12]:
def compare(df, col, group_col="condition", groups=("Patient-Centred", "Task-Centred")):
    a = df.loc[df[group_col] == groups[0], col].dropna()
    b = df.loc[df[group_col] == groups[1], col].dropna()
    if len(a) < 2 or len(b) < 2:
        return None
    t, p_t = stats.ttest_ind(a, b, equal_var=False)
    try:
        uu, p_u = stats.mannwhitneyu(a, b, alternative="two-sided")
        rb = 1 - 2 * uu / (len(a) * len(b))
    except ValueError:
        p_u, rb = np.nan, np.nan
    n1, n2 = len(a), len(b)
    sp = math.sqrt(((n1-1)*a.var(ddof=1) + (n2-1)*b.var(ddof=1)) / (n1+n2-2))
    d  = (a.mean() - b.mean()) / sp if sp > 0 else np.nan
    g  = d * (1 - 3 / (4*(n1+n2) - 9)) if not np.isnan(d) else np.nan
    return {"feature": col,
            f"{groups[0]} M": a.mean(), f"{groups[0]} SD": a.std(ddof=1),
            f"{groups[1]} M": b.mean(), f"{groups[1]} SD": b.std(ddof=1),
            "t": t, "p_welch": p_t, "p_mwu": p_u, "hedges_g": g, "rank_biserial": rb}


def holm(pvals):
    p = np.asarray(pvals, float)
    ok = ~np.isnan(p)
    out = np.full_like(p, np.nan)
    idx = np.argsort(p[ok]); m = ok.sum()
    adj, run = np.empty(m), 0.0
    for r, i in enumerate(idx):
        run = max(run, (m - r) * p[ok][i])
        adj[i] = min(run, 1.0)
    out[ok] = adj
    return out


DOSAGE = ["assistant_words", "n_assistant_turns", "mean_words_per_turn",
          "session_minutes", "n_questions", "n_user_turns", "user_words",
          "fk_grade", "type_token_ratio"]

ENOUGH = features.groupby("condition").PID.nunique().min() >= 2 and features.condition.nunique() >= 2

dose = pd.DataFrame([r for r in (compare(features, c) for c in DOSAGE) if r])
if len(dose):
    dose["p_holm"] = holm(dose.p_welch)
    print(dose.round(3).to_string(index=False))
    sig = dose[dose.p_holm < .05].feature.tolist()
    print("\nDIFFER BETWEEN CONDITIONS -> covary these in the H2 models:", sig if sig else "none")
else:
    print("Too few participants per condition for tests. Descriptives only:\n")
    print(features.groupby("condition")[DOSAGE].mean().round(2).T)

            feature  Patient-Centred M  Patient-Centred SD  Task-Centred M  Task-Centred SD       t  p_welch  p_mwu  hedges_g  rank_biserial  p_holm
    assistant_words            392.302             110.678         326.755           74.754   3.573    0.001  0.000     0.689         -0.406   0.003
  n_assistant_turns             15.283               3.639          14.057            2.735   1.961    0.053  0.071     0.378         -0.202   0.177
mean_words_per_turn             25.578               3.110          23.226            2.451   4.325    0.000  0.000     0.834         -0.506   0.000
    session_minutes              9.752               1.637           9.432            1.417   1.074    0.285  0.303     0.207         -0.116   0.571
        n_questions             12.830               2.953          10.906            2.115   3.857    0.000  0.000     0.744         -0.440   0.001
       n_user_turns             14.189               3.848          12.698            2.736   2.298    0.0

# style features
PCC > TCC

In [13]:
STYLE = PC_COLS + TC_COLS + ["patient_echo_rate", "PC_index", "TC_index", "style_index"]

EXPECTED = dict([(c, "PC") for c in PC_COLS] + [(c, "TC") for c in TC_COLS] +
                [("patient_echo_rate", "PC"), ("PC_index", "PC"),
                 ("TC_index", "TC"), ("style_index", "PC")])

style_res = pd.DataFrame([r for r in (compare(features, c) for c in STYLE) if r])
if len(style_res):
    style_res["p_holm"] = holm(style_res.p_welch)
    style_res["expected_higher_in"] = style_res.feature.map(EXPECTED)
    style_res["direction_ok"] = np.where(
        style_res.expected_higher_in == "PC", style_res.hedges_g > 0, style_res.hedges_g < 0)
    print(style_res.round(3).to_string(index=False))

    print("\n--- Verdict ---")
    n_ok = ((style_res.p_holm < .05) & style_res.direction_ok).sum()
    print(f"{n_ok}/{len(style_res)} features significant AND in the predicted direction.")
    bad = style_res[(style_res.p_holm < .05) & (~style_res.direction_ok)]
    if len(bad):
        print("Significant in the WRONG direction — report these:", bad.feature.tolist())
else:
    print("Too few participants per condition for tests. Descriptives only:\n")
    desc = features.groupby("condition")[STYLE].mean().round(2).T
    desc["expected_higher_in"] = desc.index.map(EXPECTED)
    print(desc)

             feature  Patient-Centred M  Patient-Centred SD  Task-Centred M  Task-Centred SD       t  p_welch  p_mwu  hedges_g  rank_biserial  p_holm expected_higher_in  direction_ok
       first_sg_p100              3.155               0.766           1.431            0.795  11.375      0.0    0.0     2.194         -0.857     0.0                 PC          True
       first_pl_p100              1.635               0.770           0.391            0.288  11.016      0.0    0.0     2.125         -0.905     0.0                 PC          True
  second_person_p100              7.186               0.984           4.311            0.974  15.117      0.0    0.0     2.915         -0.957     0.0                 PC          True
         affect_p100              4.600               0.975           1.641            0.666  18.244      0.0    0.0     3.518         -0.986     0.0                 PC          True
       ack_warm_p100              1.275               0.503           0.143          

# distinctive terms


In [14]:
def log_odds_dirichlet(counts_a, counts_b, alpha=0.01):
    vocab = set(counts_a) | set(counts_b)
    na, nb = sum(counts_a.values()), sum(counts_b.values())
    a0 = alpha * len(vocab)
    out = {}
    for w in vocab:
        ya, yb = counts_a.get(w, 0), counts_b.get(w, 0)
        if ya + yb < 5:
            continue
        la = math.log((ya + alpha) / (na + a0 - ya - alpha))
        lb = math.log((yb + alpha) / (nb + a0 - yb - alpha))
        var = 1/(ya + alpha) + 1/(yb + alpha)
        out[w] = (la - lb) / math.sqrt(var)
    return pd.Series(out).sort_values(ascending=False)


def cond_counts(cond):
    pids = features.loc[features.condition == cond, "PID"]
    sub = turns_df[(turns_df.PID.isin(pids)) & (turns_df.role == "assistant") & (~turns_df.is_fixed)]
    return Counter(w for t in sub.text for w in tokens(t) if w not in STOPWORDS)

conds = [c for c in ("Patient-Centred", "Task-Centred") if c in set(features.condition)]
if len(conds) == 2:
    lo = log_odds_dirichlet(cond_counts(conds[0]), cond_counts(conds[1]))
    dt = pd.DataFrame({"term": list(lo.index), "z": lo.values})
    dt["characteristic_of"] = np.where(dt.z > 0, conds[0], conds[1])
    print(f"Most characteristic of {conds[0]}:\n", lo.head(25).round(2).to_string())
    print(f"\nMost characteristic of {conds[1]}:\n", lo.tail(25).round(2).to_string())
else:
    dt = pd.DataFrame()
    print("Need both conditions present to compute distinctive terms.")

Most characteristic of Patient-Centred:
 thank         9.27
like          8.73
anything      7.53
know          6.92
let           6.69
you'd         6.34
glad          6.04
further       5.50
i'm           5.48
hear          5.42
you're        5.26
letting       5.24
explain       5.09
explore       5.03
else          4.27
feeling       4.11
help          3.93
take          3.93
sharing       3.74
understand    3.67
feel          3.48
consider      3.40
good          3.40
care          3.38
out           3.36

Most characteristic of Task-Centred:
 ready          -2.96
recommend      -3.34
several        -3.51
such           -3.73
including      -3.79
conditions     -3.97
medical        -4.18
well           -4.30
symptoms       -4.37
them           -4.71
discuss        -4.74
follows        -4.92
recurrence     -5.01
scan           -5.09
wish           -5.15
information    -5.36
consent        -5.67
prevent        -5.72
see            -5.76
confirmed      -5.81
diagnosis      -6.77
note

# export

In [15]:
features.to_csv(f"{OUT_DIR}/manipulation_check_features.csv", index=False)
content.to_csv(f"{OUT_DIR}/content_equivalence.csv", index=False)
if len(dt):
    dt.to_csv(f"{OUT_DIR}/distinctive_terms.csv", index=False)

LABELS = {
    "assistant_words": "Assistant words (total)", "n_assistant_turns": "Assistant turns",
    "mean_words_per_turn": "Words per turn", "session_minutes": "Consultation length (min)",
    "n_questions": "Questions asked", "fk_grade": "Flesch--Kincaid grade",
    "first_sg_p100": "First-person singular", "first_pl_p100": "First-person plural",
    "second_person_p100": "Second person", "affect_p100": "Affective vocabulary",
    "ack_warm_p100": "Affiliative acknowledgement", "hedge_p100": "Hedges / mitigated requests",
    "emotion_q_p100": "Emotion-eliciting questions", "personalisation_p100": "Personalisation markers",
    "patient_echo_rate": "Patient-word echo rate", "passive_p100": "Agentless / passive constructions",
    "ack_procedural_p100": "Procedural acknowledgement", "directive_p100": "Bare directives",
    "technical_p100": "Technical register", "style_index": "Composite style index (PC $-$ TC)",
}

def to_latex(res, caption, label):
    lines = [r"\begin{table}[t]", r"\centering", r"\small",
             r"\begin{tabular}{lrrrrr}", r"\toprule",
             r"Feature & PC $M$ (SD) & TC $M$ (SD) & $t$ & $p_{\text{Holm}}$ & $g$ \\",
             r"\midrule"]
    for _, r in res.iterrows():
        nm = LABELS.get(r.feature, r.feature.replace("_", r"\_"))
        p = "$<$.001" if r.p_holm < .001 else f"{r.p_holm:.3f}".lstrip("0")
        lines.append(f"{nm} & {r['Patient-Centred M']:.2f} ({r['Patient-Centred SD']:.2f}) & "
                     f"{r['Task-Centred M']:.2f} ({r['Task-Centred SD']:.2f}) & "
                     f"{r.t:.2f} & {p} & {r.hedges_g:.2f} \\\\")
    lines += [r"\bottomrule", r"\end{tabular}",
              rf"\caption{{{caption}}}", rf"\label{{{label}}}", r"\end{table}"]
    return "\n".join(lines)

if len(dose) and len(style_res):
    tex = to_latex(dose, "Dosage measures by condition. Rates are per consultation.",
                   "tab:dosage") + "\n\n" + \
          to_latex(style_res[style_res.feature.isin(PC_COLS + TC_COLS + ['style_index'])],
                   "Manipulation check. Linguistic marker rates per 100 assistant words. "
                   "$p$ values Holm-corrected within family.", "tab:manipcheck")
    with open(f"{OUT_DIR}/manipulation_check_table.tex", "w") as f:
        f.write(tex)
    print(tex[:1200])
else:
    print("Skipped LaTeX export — not enough participants per condition.")
print("\nWrote:", os.listdir(OUT_DIR))

\begin{table}[t]
\centering
\small
\begin{tabular}{lrrrrr}
\toprule
Feature & PC $M$ (SD) & TC $M$ (SD) & $t$ & $p_{\text{Holm}}$ & $g$ \\
\midrule
Assistant words (total) & 392.30 (110.68) & 326.75 (74.75) & 3.57 & .003 & 0.69 \\
Assistant turns & 15.28 (3.64) & 14.06 (2.73) & 1.96 & .177 & 0.38 \\
Words per turn & 25.58 (3.11) & 23.23 (2.45) & 4.33 & $<$.001 & 0.83 \\
Consultation length (min) & 9.75 (1.64) & 9.43 (1.42) & 1.07 & .571 & 0.21 \\
Questions asked & 12.83 (2.95) & 10.91 (2.11) & 3.86 & .001 & 0.74 \\
n\_user\_turns & 14.19 (3.85) & 12.70 (2.74) & 2.30 & .119 & 0.44 \\
user\_words & 81.32 (49.18) & 61.70 (49.97) & 2.04 & .177 & 0.39 \\
Flesch--Kincaid grade & 6.23 (0.82) & 7.83 (0.71) & -10.71 & $<$.001 & -2.06 \\
type\_token\_ratio & 0.53 (0.06) & 0.53 (0.06) & -0.28 & .780 & -0.05 \\
\bottomrule
\end{tabular}
\caption{Dosage measures by condition. Rates are per consultation.}
\label{tab:dosage}
\end{table}

\begin{table}[t]
\centering
\small
\begin{tabular}{lrrrrr}
\top